# 预处理数据 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

区分场景

*  训练：**特征 + 标签**一起前向，输出预测值 + 损失；
*  推理（预测）：若无损失计算需求，可不用传入标签。

简单总结：标签张量作为真值配套存进编码数据结构；训练时前向流程依赖标签完成损失计算。

下面是用模型中心的数据在 PyTorch 上训练句子分类器的一个例子：

In [ ]:
import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification

# Same as before
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new

#给编码字典新增标签张量，代表两条样本对应的分类标签；
#模型前向传播时需要标签才能计算损失。
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
#把 batch 字典参数解包传入模型；模型前向传播，自动计算分类损失（需要 batch 中包含标签 labels）
loss = model(**batch).loss
#反向传播，计算所有参数梯度。
loss.backward()
#优化器根据梯度更新模型权重，完成一轮参数更新。
optimizer.step()

##从模型中心（Hub）加载数据集

🤗 Datasets 库提供了一条非常便捷的命令`load_dataset`，可以在模型中心（hub）上下载和缓存数据集。你可以以下代码下载 MRPC 数据集：

（现在让我们使用 MRPC 数据集中的 GLUE 基准测试数据集作为我们训练所使用的数据集）

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

现在我们获得了一个 `DatasetDict` 对象，这个对象包含训练集、验证集和测试集。每一个集合都包含 4 个列（ sentence1 ， sentence2 ， label 和 idx ）以及一个代表行数的变量（每个集合中的行的个数）。

我们可以访问该数据集中的每一个 raw_train_dataset 对象，例如使用字典

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'idx': 0,
 'label': 1,
 'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .'}

如果想要知道不同**数字对应标签的实际含义**，我们可以查看 raw_train_dataset 的 `features` 。这告诉我们每列的类型：

In [ ]:
raw_train_dataset.features

{'sentence1': Value(dtype='string', id=None),
 'sentence2': Value(dtype='string', id=None),
 'label': ClassLabel(num_classes=2, names=['not_equivalent', 'equivalent'], names_file=None, id=None),
 'idx': Value(dtype='int32', id=None)}

##预处理数据集

为了预处理数据集，我们需要将文本转换为模型能够理解的数字。在 第二章 我们已经学习过。这是通过一个 Tokenizer 完成的，我们可以向 Tokenizer 输入一个句子或一个句子列表。以下代码表示对每对句子中的所有第一句和所有第二句进行 tokenize：

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_sentences_1 = tokenizer(raw_datasets["train"]["sentence1"])
tokenized_sentences_2 = tokenizer(raw_datasets["train"]["sentence2"])

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

{ 
  'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}

我们在 第二章 讨论了 `input_ids` 和 `attention_mask` ，但尚未讨论 token类型ID(token_type_ids) 。在本例中， **`token_type_ids` 的作用就是告诉模型输入的哪一部分是第一句，哪一部分是第二句。**

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

['[CLS]', 'this', 'is', 'the', 'first', 'sentence', '.', '[SEP]', 'this', 'is', 'the', 'second', 'one', '.', '[SEP]']

请注意，如果选择其他的 checkpoint，不一定具有 token_type_ids ，比如，DistilBERT 模型就不会返回。

`Tokenizer` 处理整个数据集：提供一对句子，第一个参数是它第一个句子的列表，第二个参数是第二个句子的列表。这也与我们在 第二章 中看到的填充和截断选项兼容。**预处理训练数据集的一种方法如下**：

这种方法虽然有效，但有一个缺点是它返回的是一个字典。这意味着在转换过程中要有足够的内存来存储整个数据集才不会出错。

In [ ]:
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"],
    raw_datasets["train"]["sentence2"],
    padding=True,
    truncation=True,
)

我们将使用 `Dataset.map()` 方法**将数据保存为 dataset 格式**，如果我们需要做更多的预处理而不仅仅是 tokenization 它还支持了一些额外的自定义的方法。 `map()` 方法的工作原理是**使用一个函数处理数据集的每个元素**。让我们定义一个对输入进行 tokenize 的函数：

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

该函数接收一个字典（与 dataset 的项类似）并返回一个包含 输入词id(input_ids) ， 注意力遮罩(attention_mask) 和 token_type_ids 键的新字典。请注意，如果 example 字典所对应的值包含多个句子（每个键作为一个句子列表），那么它依然可以运行。

在构建 batch 的时候。这样我们只需要填充到每个 batch 中的最大长度，而不是整个数据集的最大长度。当输入长度不稳定时，这可以节省大量时间和处理能力！

下面是我们如何使用一次性 `tokenize_function` 处理整个数据集。我们在调用 map 时使用了 `batch =True` ，这样**函数就可以同时处理数据集的多个元素，而不是分别处理每个元素，这样可以更快进行预处理。**

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['attention_mask', 'idx', 'input_ids', 'label', 'sentence1', 'sentence2', 'token_type_ids'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['attention_mask', 'idx', 'input_ids', 'label', 'sentence1', 'sentence2', 'token_type_ids'],
        num_rows: 408
    })
    test: Dataset({
        features: ['attention_mask', 'idx', 'input_ids', 'label', 'sentence1', 'sentence2', 'token_type_ids'],
        num_rows: 1725
    })
})

我们的 tokenize_function 返回包含 `input_ids`， `attention_mask` 和 `token_type_ids` 键的字典，这三个字段被添加到数据集的三个集合里（训练集、验证集和测试集）。

##动态填充
我们最后需要做的是将**所有示例填充到该 batch 中最长元素的长度**，这种技术被称为动态填充。

🤗transformer 库通过 `DataCollatorWithPadding` 为我们提供了这样一个函数，该函数会将每个 batch 句子填充到正确的长度。当你实例化它时，它需要一个 tokenizer （用来知道使用哪种填充 token 以及模型期望在输入的左边填充还是右边填充），**然后它会自动完成所有需要的操作：**

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

查看一个 batch 中每个条目的长度：

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

不出所料，我们得到了不同长度的样本，如果没有动态填充，所有的样本都必须填充到整个数据集中的最大长度，或者模型可以接受的最大长度。
让我们再次检查 `data_collator` 是否正确地动态填充了这批样本：

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'attention_mask': torch.Size([8, 67]),
 'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'labels': torch.Size([8])}